In [ ]:
# Cell 1 — environment, paths, and explicit ERA5 evidence (edit here only).
from pathlib import Path
import hashlib, json, re, platform
import numpy as np
import pandas as pd
import xarray as xr
import cfgrib
import eccodes

REFERENCE_PR = "https://github.com/ThemisZepplin/fig6_results/pull/14"
REFERENCE_COMMIT = "544c901"  # reviewed source reference, NEVER an HPC Git HEAD
V12 = Path("/data1/huangy/fig6/NC/v12/four_init_full_pipeline_NCVIconcurrent")
STAGE_TABLE = V12 / "tables/table_ncvi_concurrent_stage_values.csv"
DESIGN_TABLE = V12 / "mlr_raw_residual_comparison/tables/table_mlr_design_members.csv"
MODEL_MANIFEST = V12 / "mlr_raw_residual_comparison/qc/comparison_manifest.json"
REALTIME_PF = Path("/data1/huangy/fig6/NC/MSE/ecmf_pf_60_2023-06.grib")
REALTIME_CF = Path("/data1/huangy/fig6/NC/cf_2023-6/ecmf_cf_60_2023-06.grib")
ERA5_CSV = Path("/data1/huangy/fig6/NC/MSE/ERA5/NCVI_theta320/processed/era5_ncvi_theta320_daily_anomaly_2023.csv")
# Existing independently established ERA5 evidence only. Missing evidence means FAIL.
# Populate from its actual audit; do NOT infer a climatology from the filename.
ERA5_EVIDENCE = {
    "csv_sha256": None, "units": None, "level_K": None,
    "climatology_sources": [], "baseline_period": None,
}
OUTPUT_ROOT = V12 / "ncvi_daily_standalone/run_001"  # must be a new directory
INITS = ('2023-06-08', '2023-06-12')
DATES = pd.date_range('2023-06-14', '2023-06-24')
ATOL = 1e-8
CODE_SNAPSHOTS = {}
STATE = {"preflight": False, "source": False, "computed": False}

def capture_cell(number):
    # IPython captures exactly what was pasted/executed, not repository source.
    ip = get_ipython()
    text = ip.history_manager.input_hist_raw[-1]
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError("Executed cell text unavailable; audit snapshot required")
    CODE_SNAPSHOTS[number] = text
    for key in list(CODE_SNAPSHOTS):
        if key > number: del CODE_SNAPSHOTS[key]

capture_cell(1)
if OUTPUT_ROOT.exists(): raise ValueError("Output directory exists; choose a new run name")
print("Cell 1 READY", {"python": platform.python_version(), "xarray": xr.__version__, "output": str(OUTPUT_ROOT), "reference_commit": REFERENCE_COMMIT, "execution_head": "not used"})


In [ ]:
# Cell 2 — read-only GRIB metadata, no output/config file required.
capture_cell(2)
STATE.update(preflight=False, source=False, computed=False)
def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
def inspect_grib(path, init, hindcast):
    import xarray as xr
    path=Path(path).resolve(strict=True)
    with xr.open_dataset(path,engine='cfgrib',backend_kwargs={'indexpath':'','filter_by_keys':{'shortName':'pv','typeOfLevel':'theta','level':320}}) as ds:
        a=ds['pv']; refs=pd.DatetimeIndex(np.atleast_1d(a.time.values))
        matches=np.where(refs.normalize()==pd.Timestamp(init))[0]
        selected=refs[[matches[0]]] if matches.size else refs
        if not hindcast and len(matches)!=1:raise ValueError('Realtime init must match uniquely')
        record={'path':str(path),'sha256':digest(path),'selected_reference_times':[str(x) for x in selected]}
        audit=dict(record,shortName=a.attrs.get('GRIB_shortName'),typeOfLevel=a.attrs.get('GRIB_typeOfLevel'),
                   units=a.attrs.get('units'),level_coordinates={k:np.asarray(a[k]).tolist() for k in ['theta','potentialTemperature'] if k in a.coords},
                   forecast_reference_times=[str(x) for x in refs],selected_years=sorted(set(selected.year.tolist())),
                   dimensions=dict(a.sizes),coordinates={k:[str(v) for v in np.atleast_1d(a[k].values)] for k in ['step','number','time'] if k in a.coords})
        if audit['shortName']!='pv' or audit['typeOfLevel']!='theta':raise ValueError('Uncertain GRIB variable/level')
        return record,audit

config = {"output_root": str(OUTPUT_ROOT), "stage_table": str(STAGE_TABLE),
          "design_table": str(DESIGN_TABLE), "comparison_manifest": str(MODEL_MANIFEST), "sources": {}}
ref = pd.read_csv(STAGE_TABLE)
metadata_audit = []
for init in INITS:
    selected = ref[(ref.init_date == init) & ref.stage.isin(['Stage-I_dry','Stage-I','Total'])]
    names = selected.climatology_file_used.unique()
    if len(names) != 1: raise ValueError(f"{init}: ambiguous/missing MLR climatology files")
    match = re.fullmatch(r'pf=(.+); cf=(.+)', names[0])
    if not match: raise ValueError("Cannot parse actual MLR reforecast paths")
    paths = {'realtime': {'pf': str(REALTIME_PF), 'cf': str(REALTIME_CF)},
             'reforecast': {'pf': match[1], 'cf': match[2]}}
    config['sources'][init] = {}
    for kind, pair in paths.items():
        config['sources'][init][kind] = {}
        for role, path in pair.items():
            item, audit = inspect_grib(path, init, kind == 'reforecast')
            item['path'] = path  # exact recorded pathname; audit contains resolved target
            config['sources'][init][kind][role] = item
            metadata_audit.append(dict(audit, init_date=init, kind=kind, role=role,
                                       mlr_climatology_file_used=names[0]))
STATE['preflight'] = True
print("Cell 2 metadata PASS; review actual selection below")
for a in metadata_audit:
    print({k: a[k] for k in ('init_date','kind','role','path','sha256','shortName','typeOfLevel','level_coordinates','units','dimensions','selected_reference_times','selected_years')})


In [ ]:
# Cell 3 — hashes, strict field/member/sample checks and independent ERA5 QC.
capture_cell(3)
STATE.update(source=False, computed=False)
if not STATE['preflight']: raise RuntimeError("Run Cell 2 successfully first")
def load_pair(spec, init, hindcast=False):
    """Open explicitly pinned GRIB files only; validate level/units/member identity.

    Reforecast selector mirrors MLR: first exact init-date match, otherwise all
    records. The retained forecast-reference list must exactly match config.
    Nonmatching month/day or hour rejects the MLR fallback rather than using it.
    """
    import xarray as xr
    parts=[];audit=[]
    for role in ['cf','pf']:
        item=spec[role];path=Path(item['path'])
        if digest(path)!=item['sha256']:
            raise ValueError('GRIB source hash differs from pinned config')
        with xr.open_dataset(path,engine='cfgrib',backend_kwargs={'indexpath':'','filter_by_keys':{'shortName':'pv','typeOfLevel':'theta','level':320}}) as ds:
            if 'pv' not in ds:
                raise ValueError('Actual variable must be pv')
            a=ds.pv
            if a.attrs.get('GRIB_typeOfLevel')!='theta':
                raise ValueError('Require proven isentropic theta level')
            level=next((c for c in ['theta','potentialTemperature'] if c in a.coords),None)
            if level is None or np.size(a[level])!=1 or float(np.asarray(a[level]).ravel()[0])!=320:
                raise ValueError('Require exact 320 K; no nearest-layer substitution')
            if level in a.dims:a=a.squeeze(level,drop=True)
            units=str(a.attrs.get('units',''));normalized=units.lower().replace(' ','').replace('**','^')
            if normalized not in ['km^2kg^-1s^-1','k*m^2*kg^-1*s^-1','pvu']:
                raise ValueError(f'Unknown PV units: {units}; no magnitude heuristic')
            factor=1. if normalized=='pvu' else 1e6
            if 'time' not in a.coords:raise ValueError('Missing forecast-reference time')
            refs=pd.DatetimeIndex(np.atleast_1d(a.time.values))
            matches=np.where(refs.normalize()==pd.Timestamp(init))[0]
            if not hindcast and len(matches)!=1:raise ValueError('Realtime init not unique')
            if matches.size:
                chosen=refs[[matches[0]]]
                if 'time' in a.dims:a=a.isel(time=int(matches[0]),drop=True)
            else:
                if not hindcast:raise ValueError('Missing realtime init')
                chosen=refs
            if [str(x) for x in chosen]!=item['selected_reference_times']:
                raise ValueError('Selected reference-time sample set differs from explicit config')
            if any((t.month,t.day,t.hour,t.minute)!=(pd.Timestamp(init).month,pd.Timestamp(init).day,0,0) for t in chosen):
                raise ValueError('Reforecast init-day/hour mismatch')
            if role=='cf':
                if 'number' in a.dims:
                    if a.sizes['number']!=1:raise ValueError('CF must have one member')
                    a=a.assign_coords(number=[0])
                else:a=a.expand_dims(number=[0])
            elif 'number' not in a.dims or not np.array_equal(a.number.values,np.arange(1,11 if hindcast else 51)):
                raise ValueError('PF IDs must be 1..10 for reforecast or 1..50 for realtime; no renumbering')
            lat=a.latitude;lon=a.longitude
            a=a.sel(latitude=lat[(lat>=35)&(lat<=50)],longitude=lon[(lon>=115)&(lon<=130)])
            if a.sizes['latitude']==0 or a.sizes['longitude']==0 or not np.isfinite(a.values).all():
                raise ValueError('Empty region/missing grid values; no silent skip')
            attrs=dict(a.attrs)
            regional=(a.mean(['latitude','longitude'],skipna=False)*factor).load()
            parts.append(regional)
            audit.append(dict(init_date=init,role=('reforecast_' if hindcast else 'realtime_')+role,
                              path=str(path),sha256=item['sha256'],variable='pv',level=320,units=units,
                              pvu_multiplier=factor,reference_times=json.dumps([str(t) for t in chosen]),
                              sample_coordinates=json.dumps({d:[str(x) for x in regional[d].values] for d in regional.dims if d not in ['step','number']}),
                              attributes=json.dumps(attrs,default=str),spatial_mean='arithmetic grid-cell mean',
                              legacy_spatial_mean='cos(latitude) weighted',region='35-50N 115-130E'))
    merged=xr.concat(parts,dim='number',join='exact',compat='equals').sortby('number')
    if not np.array_equal(merged.number.values,np.arange(11 if hindcast else 51)):
        raise ValueError('Require CF0+PF1..10 reforecast or CF0+PF1..50 realtime')
    if hindcast:
        if set(merged.dims)!={'time','step','number'} or merged.sizes['time']!=20:
            raise ValueError('Reforecast requires retained time x step x 11 members, 20 years')
        refs=pd.DatetimeIndex(merged.time.values)
        if refs.has_duplicates or sorted(refs.year.tolist())!=list(range(2003,2023)):
            raise ValueError('Reforecast years must be 2003..2022 exactly once')
        extra=['time','number']
        merged=merged.stack(sample=extra).transpose('step','sample')
    else:
        if set(merged.dims)!={'step','number'}:raise ValueError('Unexpected realtime dimensions')
        merged=merged.transpose('step','number')
    return merged.values,merged.step.values,audit

model_manifest = json.loads(MODEL_MANIFEST.read_text())
if digest(STAGE_TABLE) != next(p['sha256'] for p in model_manifest['provenance'] if p['name']=='ncvi'):
    raise ValueError("Stage source hash mismatch")
if digest(DESIGN_TABLE) != model_manifest['table_sha256']['table_mlr_design_members.csv']:
    raise ValueError("Design source hash mismatch")
ref['window'] = ref.stage.replace({'Stage-I_dry':'Stage-I'})
ref = ref[ref.init_date.isin(INITS) & ref.window.isin(['Stage-I','Total'])].copy()
if len(ref)!=204 or ref.duplicated(['init_date','window','member']).any() or not ref.anomaly_used.eq(True).all() or ref.fallback_reason.fillna('').ne('').any():
    raise ValueError("Require 204 unique stage anomaly references without fallback")
design = pd.read_csv(DESIGN_TABLE)
q = design[(design.predictor=='NCVI_concurrent') & design.init_date.isin(INITS) & design.window.isin(['Stage-I','Total'])].merge(ref,on=['init_date','window','member'],validate='many_to_one')
if len(q)!=408 or q.version.nunique()!=2 or not np.allclose(q.value,q.NCVI_concurrent,rtol=0,atol=ATOL):
    raise ValueError("MLR design/source mismatch")
source_arrays = {}; source_audit = []
for init in INITS:
    spec = config['sources'][init]
    for role, suffix in [('pf','MSE/ecmf_pf_60_2023-06.grib'),('cf','cf_2023-6/ecmf_cf_60_2023-06.grib')]:
        if not str(spec['realtime'][role]['path']).endswith(suffix):
            raise ValueError("Realtime path differs from preserved MLR")
    rt, rs, ra = load_pair(spec['realtime'], init)
    hc, hs, ha = load_pair(spec['reforecast'], init, True)
    source_arrays[init] = (rt, rs, hc, hs)
    source_audit.extend(ra+ha)

# ERA5 is independent. Only verified existing provenance is accepted.
era_qc = {'status':'FAIL','reason':'ERA5 evidence not checked'}
try:
    era = pd.read_csv(ERA5_CSV)
    era = era[pd.to_datetime(era.date).isin(DATES)]
    ep = ERA5_EVIDENCE
    if len(era)!=11 or era.date.duplicated().any() or not np.isfinite(era.ncvi_theta320_anom_pvu).all() or ep['csv_sha256']!=digest(ERA5_CSV) or ep['units']!='PVU' or ep['level_K']!=320 or not ep['climatology_sources'] or not ep['baseline_period']:
        raise ValueError('ERA5 coverage/climatology provenance invalid or incomplete')
    era_qc = {'status':'PASS','reason':'Existing CSV coverage and supplied independent provenance verified','fields':list(era.columns)}
except (OSError,ValueError,KeyError,TypeError,AttributeError) as exc:
    era_qc = {'status':'FAIL','reason':str(exc)}
STATE['source'] = True
print("Cell 3 S2S source PASS; 204 stage refs / 408 model refs verified; CF0+PF1–50 and pinned samples validated")
print("ERA5 reference", era_qc)


In [ ]:
# Cell 4 — unchanged scientific daily computation/replay function.
capture_cell(4)
STATE['computed'] = False
if not STATE['source']: raise RuntimeError("Run Cell 3 successfully first")
def daily_replay(init, rt, rt_steps, hc, hc_steps, reference):
    """Inputs are regional PVU: realtime[step,51], climatology[step,sample].

    Stage scalars use equal effective-time/sample weights, exactly as preserved
    MLR; daily anomaly remains unadjusted. Both equal-day and time-weighted
    replays are retained. Variable timestep counts cannot be hidden by rescaling.
    """
    if init not in INITS:
        raise ValueError('Unsupported init')
    arrays = [np.asarray(rt,float), np.asarray(hc,float)]
    if arrays[0].ndim != 2 or arrays[0].shape[1] != 51 or arrays[1].ndim != 2:
        raise ValueError('Require realtime 51 members and explicit hindcast samples')
    if not all(np.isfinite(x).all() for x in arrays):
        raise ValueError('Missing PV values are forbidden')
    rt,hc = arrays
    times=[]
    for steps, values in [(rt_steps,rt),(hc_steps,hc)]:
        steps=np.asarray(steps)
        if not np.issubdtype(steps.dtype,np.timedelta64):
            raise ValueError('Lead coordinates must be explicit timedeltas')
        t=pd.DatetimeIndex(pd.Timestamp(init)+pd.to_timedelta(steps)+pd.Timedelta(hours=8))
        if len(t)!=len(values) or t.has_duplicates or not t.is_monotonic_increasing:
            raise ValueError('Duplicated/unsorted or inconsistent valid times')
        times.append(t)
    rtime,htime=times
    rows=[];clims=[]
    for date in DATES:
        rmask=rtime.normalize()==date;hmask=htime.normalize()==date
        if not rmask.any() or not hmask.any():
            raise ValueError(f'{init}/{date}: missing realtime/reforecast daily coverage')
        clim=float(hc[hmask].mean());real=rt[rmask].mean(axis=0)
        clims.append(dict(init_date=init,date=str(date.date()),daily_climatology_pvu=clim,
                          realtime_time_count=int(rmask.sum()),reforecast_time_count=int(hmask.sum()),
                          reforecast_sample_count=hc.shape[1],reforecast_value_count=int(hmask.sum()*hc.shape[1])))
        for member in range(51):
            rows.append(dict(init_date=init,date=str(date.date()),member=member,realtime_pv_pvu=real[member],
                             daily_climatology_pvu=clim,daily_ncvi_anomaly_pvu=real[member]-clim,
                             units='PVU',anomaly_used=True,source_qc_reference=f'{init}:source_audit',
                             realtime_time_count=int(rmask.sum()),reforecast_time_count=int(hmask.sum())))
    daily=pd.DataFrame(rows);qc=[]
    for window,end in [('Stage-I','2023-06-17'),('Total','2023-06-24')]:
        rm=(rtime.normalize()>=DATES[0])&(rtime.normalize()<=pd.Timestamp(end))
        hm=(htime.normalize()>=DATES[0])&(htime.normalize()<=pd.Timestamp(end))
        scalar=float(hc[hm].mean());replay=rt[rm].mean(axis=0)-scalar
        q=daily[daily.date<=end];equal=q.groupby('member').daily_ncvi_anomaly_pvu.mean()
        weighted=q.assign(term=q.daily_ncvi_anomaly_pvu*q.realtime_time_count).groupby('member').agg(term=('term','sum'),count=('realtime_time_count','sum'))
        targets=reference[(reference.init_date==init)&(reference.window==window)].set_index('member')
        if len(targets)!=51 or set(targets.index)!=set(range(51)):
            raise ValueError('Incomplete stage reference members')
        for member in range(51):
            target=float(targets.loc[member,'NCVI_concurrent']);w=float(weighted.loc[member,'term']/weighted.loc[member,'count'])
            qc.append(dict(init_date=init,window=window,member=member,model_stage_ncvi=target,
                           mlr_scalar_replay=replay[member],stage_climatology_scalar=scalar,
                           daily_equal_day_aggregate=equal.loc[member],daily_time_weighted_aggregate=w,
                           scalar_difference=replay[member]-target,daily_difference=equal.loc[member]-target,
                           weighted_difference=w-target,scalar_pass=abs(replay[member]-target)<=ATOL,
                           daily_pass=abs(equal.loc[member]-target)<=ATOL and abs(w-target)<=ATOL,
                           realtime_times=int(rm.sum()),reforecast_times=int(hm.sum()),
                           reforecast_samples=hc.shape[1],difference_analysis='Compare scalar replay, daily equal/time weights and per-day sample/time counts; no correction applied'))
    return daily,pd.DataFrame(clims),pd.DataFrame(qc)

print("Cell 4 functions READY; fixed tolerance", ATOL, "PVU; no model fitting")


In [ ]:
# Cell 5 — genuine computation, no CSV or figure output yet.
capture_cell(5)
STATE['computed'] = False
if not STATE['source']: raise RuntimeError("Run Cells 2–4 first")
if set(CODE_SNAPSHOTS) != {1,2,3,4,5}: raise RuntimeError("Execute Cells 1–5 in order")
daily_parts=[]; climatology_parts=[]; replay_parts=[]
for init in INITS:
    # Recheck raw files after preflight; cached arrays must still refer to pinned inputs.
    for pair in config['sources'][init].values():
        for item in pair.values():
            if digest(item['path']) != item['sha256']: raise ValueError("Source changed after preflight")
    d,c,q = daily_replay(init, *source_arrays[init], ref)
    daily_parts.append(d); climatology_parts.append(c); replay_parts.append(q)
daily = pd.concat(daily_parts,ignore_index=True)
climatology = pd.concat(climatology_parts,ignore_index=True)
replay = pd.concat(replay_parts,ignore_index=True)
if len(daily)!=1122 or daily.duplicated(['init_date','date','member']).any() or len(replay)!=204:
    raise ValueError("Unexpected daily/replay coverage")
STATE['computed'] = True
print("Cell 5 coverage PASS", {"daily_rows":len(daily),"climatology_rows":len(climatology),"stage_comparisons":len(replay)})
print("Raw maxima PVU", {name:float(replay[name].abs().max()) for name in ['scalar_difference','daily_difference','weighted_difference']})


In [ ]:
# Cell 6 — export all QC evidence; never force equivalence or overwrite outputs.
capture_cell(6)
if not STATE['computed'] or set(CODE_SNAPSHOTS)!={1,2,3,4,5,6}:
    raise RuntimeError("Run Cells 1–5 successfully and in order")
if OUTPUT_ROOT.exists(): raise ValueError("Output exists; choose a new directory in Cell 1 and restart")
# Bind the actual executed snapshots, not a presumed Git checkout.
snapshot = json.dumps(CODE_SNAPSHOTS,sort_keys=True,ensure_ascii=False,indent=2)
snapshot_hash = hashlib.sha256(snapshot.encode('utf-8')).hexdigest()
inputs = {key:{'path':str(path),'sha256':digest(path)} for key,path in
          [('stage_table',STAGE_TABLE),('design_table',DESIGN_TABLE),('comparison_manifest',MODEL_MANIFEST)]}
if ERA5_CSV.is_file(): inputs['era5_csv']={'path':str(ERA5_CSV),'sha256':digest(ERA5_CSV)}
# Model/reference files must still match the data read in Cell 3.
if inputs['stage_table']['sha256'] != next(p['sha256'] for p in model_manifest['provenance'] if p['name']=='ncvi') or inputs['design_table']['sha256'] != model_manifest['table_sha256']['table_mlr_design_members.csv']:
    raise ValueError("Model source changed during execution")
s2s_pass = bool(replay.scalar_pass.all() and replay.daily_pass.all())
formal_allowed = s2s_pass and era_qc['status']=='PASS'
status = 'PASS' if formal_allowed else ('FAIL_STAGE_EQUIVALENCE' if not s2s_pass else 'FAIL_ERA5_REFERENCE')
daily['qc_status']=status
OUTPUT_ROOT.mkdir(parents=True,exist_ok=False)
tables = {'table_ncvi_concurrent_daily_members.csv':daily,
          'table_ncvi_daily_climatology.csv':climatology,
          'table_ncvi_daily_source_audit.csv':pd.DataFrame(source_audit),
          'table_ncvi_daily_stage_replay_qc.csv':replay}
for name,table in tables.items(): table.to_csv(OUTPUT_ROOT/name,index=False)
(OUTPUT_ROOT/'executed_cell_code_snapshot.json').write_text(snapshot,encoding='utf-8')
(OUTPUT_ROOT/'ncvi_source_preflight_report.json').write_text(json.dumps({'metadata':metadata_audit,'validated_sources':source_audit,'era5_reference_qc':era_qc},indent=2,default=str))
result = {'status':status,'formal_process_allowed':formal_allowed,'atol':ATOL,
          's2s_daily_source_qc':{'status':'PASS','reason':'Pinned GRIB metadata, fields, units, members and samples validated'},
          's2s_stage_replay_qc':{'status':'PASS' if s2s_pass else 'FAIL','reason':'All 204 scalar/equal-day/time-weighted comparisons required'},
          'era5_reference_qc':era_qc,'reference_pr':REFERENCE_PR,'reference_commit':REFERENCE_COMMIT,
          'execution_git_head':None,'executed_code_snapshot_sha256':snapshot_hash,
          'config':config,'era5_evidence':ERA5_EVIDENCE,'inputs':inputs,
          'outputs':{name:digest(OUTPUT_ROOT/name) for name in tables},
          'daily_rows':len(daily),'stage_comparisons':len(replay),
          'max_scalar_difference':float(replay.scalar_difference.abs().max()),
          'max_daily_difference':float(replay.daily_difference.abs().max()),
          'max_weighted_difference':float(replay.weighted_difference.abs().max()),
          'recommendation':'If scalar replay passes but daily fails, inspect effective time counts and daily climatology sample weights; preserve all differences. Any change to MLR baseline requires separate scientific authorization. Never adjust daily anomalies.'}
result['outputs']['executed_cell_code_snapshot.json']=digest(OUTPUT_ROOT/'executed_cell_code_snapshot.json')
result['outputs']['ncvi_source_preflight_report.json']=digest(OUTPUT_ROOT/'ncvi_source_preflight_report.json')
(OUTPUT_ROOT/'ncvi_daily_export_manifest.json').write_text(json.dumps(result,indent=2,default=str))
print("Cell 6",status,"formal_process_allowed =",formal_allowed,"output =",str(OUTPUT_ROOT))
print(json.dumps({k:result[k] for k in ['s2s_daily_source_qc','s2s_stage_replay_qc','era5_reference_qc','max_scalar_difference','max_daily_difference','max_weighted_difference']},indent=2))
